## Prompt:

TASK DESCRIPTION:
This is an IMAGE-LEVEL BINARY CLASSIFICATION task implemented using an object detection model.
The goal is to determine whether an image contains an animal or not.

DATASET STRUCTURE:
DATASET_ROOT contains three subdirectories: train, test, and val.
Each directory contains two subdirectories:
images/ → contains image files (.jpg, .jpeg, .png)
labels/ → contains YOLO format .txt files

GROUND-TRUTH LOGIC: An image is considered an animal if a corresponding .txt file exists and is not empty in the labels/ folder.

MODEL REQUIREMENTS:
Use ONLY a pretrained Ultralytics YOLO detection model (e.g., yolov8n.pt).
Load the model using the Ultralytics YOLO API.
Assume YOLO detects animals using class ID animal at index 0.

DETECTION LOGIC (IMPORTANT):
Run object detection on each image.
If the model produces AT LEAST ONE detection of an animal class with confidence >= 0.5:
→ The image-level prediction is animal.

EVALUATION METRICS:
Iterate through the images in the test split.
Compare the image-level prediction with the ground truth (existence of label file).
Count: True Positives, True Negatives, False Positives, and False Negatives.

ACCURACY DEFINITION:
Overall accuracy = (True Positives + True Negatives) / Total Images

OUTPUT REQUIREMENTS:
Print for each image: filename, ground-truth status, and prediction.
At the end, print a summary report including total images, counts for each metric, and overall detection accuracy.

CODING REQUIREMENTS:
Store the main path in DATASET_ROOT.
Use pathlib or os for robust file path matching.
Read only .jpg files.
Include clear comments explaining each step.

After the code, briefly explain how the program works in plain English.

## Sample code


In [2]:
import os
from pathlib import Path
import torch
from ultralytics import YOLO

# Define the dataset root directory
DATASET_ROOT = Path('/home/jovyan/work/vista/ai-tutorial-2026/datasets/AnimalEcology.v4i.yolov11')  # Update this path to your dataset root

# Load the pretrained YOLOv8 model
model = YOLO('/home/jovyan/work/vista/ai-tutorial-2026/models/yolov9t_ep200_bs32_lr0.005_baa22147.pt')

def load_images_and_labels(test_dir):
    """
    Load image paths and their corresponding label paths.
    """
    images = []
    labels = []
    
    # Iterate over all .jpg files in the images directory
    for img_path in test_dir.glob('images/*.jpg'):
        # Construct the corresponding label file path
        label_path = test_dir / 'labels' / f"{img_path.stem}.txt"
        
        images.append(img_path)
        labels.append(label_path)
    
    return images, labels

def predict_with_yolo(model, img_path):
    """
    Perform object detection on an image using the YOLO model.
    Returns True if an animal is detected with confidence >= 0.5, else False.
    """
    results = model.predict(source=str(img_path), conf=0.5)
    for result in results:
        boxes = result.boxes.cpu().numpy()
        for box in boxes:
            cls = int(box.cls[0])
            conf = box.conf[0]
            if cls == 0 and conf >= 0.5:  # Assuming class ID 0 is for animals
                return True
    return False

def evaluate_model(images, labels):
    """
    Evaluate the model's predictions against the ground truth.
    """
    true_positives = 0
    true_negatives = 0
    false_positives = 0
    false_negatives = 0
    
    for img_path, label_path in zip(images, labels):
        # Determine ground truth
        ground_truth = label_path.exists() and label_path.stat().st_size > 0
        
        # Get model prediction
        prediction = predict_with_yolo(model, img_path)
        
        # Compare prediction with ground truth
        if ground_truth and prediction:
            true_positives += 1
        elif not ground_truth and not prediction:
            true_negatives += 1
        elif not ground_truth and prediction:
            false_positives += 1
        elif ground_truth and not prediction:
            false_negatives += 1
        
        # Print result for each image
        print(f"Filename: {img_path.name}, Ground Truth: {'Animal' if ground_truth else 'No Animal'}, Prediction: {'Animal' if prediction else 'No Animal'}")
    
    return true_positives, true_negatives, false_positives, false_negatives

def main():
    # Load test images and labels
    test_dir = DATASET_ROOT / 'test'
    images, labels = load_images_and_labels(test_dir)
    
    # Evaluate the model
    tp, tn, fp, fn = evaluate_model(images, labels)
    
    # Calculate overall accuracy
    total_images = len(images)
    accuracy = (tp + tn) / total_images
    
    # Print summary report
    print("\nSummary Report:")
    print(f"Total Images: {total_images}")
    print(f"True Positives: {tp}")
    print(f"True Negatives: {tn}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    print(f"Overall Accuracy: {accuracy:.2f}")

if __name__ == "__main__":
    main()

In [ ]:
# Define the dataset root directory
DATASET_ROOT = Path('/home/jovyan/work/vista/ai-tutorial-2026/datasets/AnimalEcology.v4i.yolov11')  # Update this path to your dataset root

# Load the pretrained YOLOv8 model
model = YOLO('/home/jovyan/work/vista/ai-tutorial-2026/models/yolov9t_ep200_bs32_lr0.005_baa22147.pt')


## Paste your generated Code here 
